# 03 — Question 2: does AI exposure improve wage prediction?

**Question:** Does adding the AI exposure score to a baseline wage model (demographics, education, region, year, sector) reduce test RMSE on individuals held out from training? If so, does the score add predictive information beyond (a) an arbitrary occupation-level score and (b) standard occupational categories?

**Rule:** no individual rows are displayed, only RMSEs, means and counts.

**Course methods:** `train_test_split` and `LinearRegression` (Week 6), `RandomForestRegressor` and `GridSearchCV` (Week 7), `get_dummies` (Week 2), RMSE (Week 6).
**Beyond the course (flagged):** `GroupKFold` for cross-validation by individual (step 6).

| Model | Variables | What it tests |
|---|---|---|
| A | Baseline: age, sex, education, region, year, sector | Reference |
| B | A + exposure | Does exposure reduce prediction error? |
| C | A + placebo (scores shuffled across occupations) | Same granularity, without the "AI" meaning |
| D | A + major ISCO group (1 digit) | Coarse occupational information |
| E | D + exposure | Does exposure add anything within major groups? |
| F | A + 2-digit ISCO | Finer occupation, no AI |
| G | F + exposure | Does exposure add anything beyond **fine** occupational categories? |

**Why model G?** The question asks whether the gain is specific to AI or comes from finer occupational detail.
Comparing E with D answers this only for the **major groups** (9 categories). A score that adds nothing beyond 9 categories might still add something beyond finer categories, and vice versa.
G vs F therefore repeats the same test with the **44 two-digit groups**: if exposure adds nothing at this level either, we can conclude that it carries no information beyond standard occupational categories, not only beyond major groups.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

DATA_PATH = "../Data/private/shp_isco4_ilo.parquet"

## 1. Load the data
We drop the 80 rows without an exposure score and the category `sex = 3` (22 rows, introduced in 2024), as planned in the cleaning.

In [ ]:
df = pd.read_parquet(DATA_PATH)
df = df[df["exposure_source"] != "none"]
df = df[df["sex"].isin([1, 2])]

print("Rows:", df.shape[0], "| Individuals:", df["idpers"].nunique())
print(df["exposure_source"].value_counts())

## 2. Prepare the variables
- `age2` = age squared: wages rise with age and then level off (concave profile). A linear regression needs this term to capture the curvature.
- Categorical variables → dummies with `get_dummies(drop_first=True)` to avoid the dummy variable trap (Week 6).
- `isco1` and `isco2`: first digits of the 4-digit ISCO code (string).

In [ ]:
df["age2"] = df["age"] ** 2
df["isco1"] = df["isco4"].str[:1]
df["isco2"] = df["isco4"].str[:2]

# Numerically coded categories -> strings, so that get_dummies treats them as categories
for col in ["sex", "isced", "region", "year", "noga2m"]:
    df[col] = df[col].astype("Int64").astype(str)

dummies = pd.get_dummies(df[["sex", "isced", "region", "year", "noga2m"]], drop_first=True, dtype=int)
d_isco1 = pd.get_dummies(df["isco1"], prefix="isco1", drop_first=True, dtype=int)
d_isco2 = pd.get_dummies(df["isco2"], prefix="isco2", drop_first=True, dtype=int)

BASE = ["age", "age2"] + list(dummies.columns)
df = pd.concat([df, dummies, d_isco1, d_isco2], axis=1)

print("Baseline variables:", len(BASE))
print("1-digit ISCO dummies:", d_isco1.shape[1], "| 2-digit ISCO dummies:", d_isco2.shape[1])

## 3. Placebo score
We keep **one value per occupation**, but **shuffle the scores across occupations**: a software developer may, for example, receive a baker's score.
The placebo therefore has exactly the same granularity as exposure, but no "AI" meaning.

In [ ]:
occupation_scores = df.groupby("isco4")["exposure_main"].first()

rng = np.random.default_rng(42)
shuffled_scores = pd.Series(rng.permutation(occupation_scores.values), index=occupation_scores.index)
df["placebo"] = df["isco4"].map(shuffled_scores)

print("Correlation exposure / placebo:", round(df["exposure_main"].corr(df["placebo"]), 3))   # close to 0

## 4. The seven feature sets

In [ ]:
MODELS = {
    "A_baseline":         BASE,
    "B_exposure":         BASE + ["exposure_main"],
    "C_placebo":          BASE + ["placebo"],
    "D_isco1":            BASE + list(d_isco1.columns),
    "E_isco1_exposure":   BASE + list(d_isco1.columns) + ["exposure_main"],
    "F_isco2":            BASE + list(d_isco2.columns),
    "G_isco2_exposure":   BASE + list(d_isco2.columns) + ["exposure_main"],
}
y = df["log_income"]

## 5. Split by individual
An individual appears up to 9 times. If their years were split between train and test, the model would "recognise" them and the test error would be too optimistic.

We therefore apply `train_test_split` (Week 6) **to the list of individuals**, then keep all rows of each individual on the same side.

In [ ]:
def split_by_individual(df, seed):
    individuals = df["idpers"].unique()
    ind_train, ind_test = train_test_split(individuals, test_size=0.2, random_state=seed)
    return df["idpers"].isin(ind_train), df["idpers"].isin(ind_test)

train, test = split_by_individual(df, seed=42)
print("Train rows:", train.sum(), "| test rows:", test.sum())
print("Individuals in both sets:", len(set(df.loc[train, "idpers"]) & set(df.loc[test, "idpers"])))   # must be 0

## 6. Choose the random forest hyperparameters
`GridSearchCV` as in Week 7, **once**, on model B and on the training set only.
The same hyperparameters are then used for all models, so that the comparison only concerns the variables.

**Beyond the course:** `cv=GroupKFold(5)` with `groups=idpers` also makes cross-validation **by individual**. With the default `cv=5`, the same individual could end up in two folds.

In [ ]:
param_grid = {"max_depth": [10, 20, None], "min_samples_leaf": [5, 20, 50]}

grid = GridSearchCV(
    RandomForestRegressor(n_estimators=100, max_features=0.5, random_state=42, n_jobs=-1),
    param_grid,
    cv=GroupKFold(n_splits=5),
    scoring="neg_root_mean_squared_error",
)
grid.fit(df.loc[train, MODELS["B_exposure"]], y[train], groups=df.loc[train, "idpers"])

print("Best hyperparameters:", grid.best_params_)
print("Cross-validation RMSE:", round(-grid.best_score_, 4))

## 7. Train and evaluate the 14 models (7 feature sets × 2 algorithms)
Metric: **test RMSE** on log wages. An RMSE of 0.30 corresponds to a typical error of about 30%.

In [ ]:
def evaluate(train, test):
    results = []
    for name, variables in MODELS.items():
        X_train, X_test = df.loc[train, variables], df.loc[test, variables]

        lr = LinearRegression().fit(X_train, y[train])
        rf = RandomForestRegressor(n_estimators=200, max_features=0.5, random_state=42,
                                   n_jobs=-1, **grid.best_params_).fit(X_train, y[train])

        results.append({"model": name,
                        "RMSE_linear": root_mean_squared_error(y[test], lr.predict(X_test)),
                        "RMSE_forest": root_mean_squared_error(y[test], rf.predict(X_test))})
    return pd.DataFrame(results).set_index("model")

res = evaluate(train, test)

# Gain relative to the baseline, in % of the RMSE
for algo in ["RMSE_linear", "RMSE_forest"]:
    res["gain_" + algo[5:] + "_%"] = (1 - res[algo] / res.loc["A_baseline", algo]) * 100

res.round(4)

## 8. Robustness: 10 different splits
A single split may give a result driven by chance (which individuals end up in the test set).
We repeat with 10 draws and look at the **mean and standard deviation of the gain**. A gain smaller than its standard deviation is not reliable.

⏱ This cell takes about 10 to 15 minutes (140 models).

In [ ]:
all_runs = []
for seed in range(10):
    tr, te = split_by_individual(df, seed)
    r = evaluate(tr, te)
    for algo in ["RMSE_linear", "RMSE_forest"]:
        r["gain_" + algo[5:] + "_%"] = (1 - r[algo] / r.loc["A_baseline", algo]) * 100
    r["seed"] = seed
    all_runs.append(r.reset_index())

all_runs = pd.concat(all_runs)
summary = all_runs.groupby("model")[["gain_linear_%", "gain_forest_%"]].agg(["mean", "std"]).round(2)
summary

## 9. How to read the results

| Comparison | If… | Then… |
|---|---|---|
| **B vs A** | B's gain > 0, clearly above its standard deviation | exposure reduces prediction error |
| **B vs C** | B ≫ C | the gain comes from the **content** of the score (its "AI" meaning), not only from its granularity |
| | B ≈ C | any occupation-level number would do as well → it is **occupational information** |
| **E vs D** | E > D | within major groups, exposure still adds something |
| | E ≈ D | exposure only summarises the major occupational group |
| **G vs F** | G > F | even with 44 occupational groups, exposure still adds something |
| | G ≈ F | exposure adds nothing beyond fine occupational categories |
| **B vs F** | F ≫ B | a finer occupation, without AI, predicts better → B's gain comes mostly from occupational detail |

**For E vs D and G vs F, read the linear model first**: the random forest hyperparameters were chosen on model B, which may favour models that include exposure.

**Caveat on the random forest and the placebo:** a forest can use any occupation-level number as an **identifier** of the occupation, whatever its meaning. The placebo C may therefore help the forest without helping the linear regression. The B vs C test is more informative with the **linear** model, where a score unrelated to wages cannot help.

**What this does not tell us:** a gain in prediction is not a causal effect. Exposed occupations are also those requiring more education, and question 2 does not say that AI *raises or lowers* wages.

## 10. Conclusion

**Answer to Q2.** Adding the AI exposure score to the baseline wage model reduces test RMSE on held-out individuals by 0.56% (SD 0.22) with linear regression and 2.51% (SD 0.42) with a random forest (mean over 10 person-level splits). The gain is not an artefact of granularity: a placebo score, which keeps one value per occupation but shuffles values across occupations, yields almost no improvement (0.02% and 0.29%).

However, the gain is occupational information rather than AI-specific information. Occupational categories alone reduce RMSE far more (4.70% with 1-digit ISCO, 5.05% with 2-digit ISCO, linear model), and once they are included, the exposure score adds nothing (+0.03 percentage points at both levels). In the random forest, exposure still adds about 1.4 to 2.0 points beyond occupational categories, but this reflects the forest's difficulty with many occupation dummies: the 2-digit model performs worse than the 1-digit model, and the best forest model (4.44%) remains below the best linear model (5.08%). Exposure therefore acts as a compact numerical summary of occupation, ordered in a way that correlates with wages.

**Interpretation.** AI exposure is a useful summary of occupation for wage prediction, but it carries no predictive information beyond standard occupational categories. This is consistent with exposure reflecting the skill content of occupations. These are predictive associations, not causal effects.

**Limitations.** (1) Overall gains are small: the best model reduces RMSE by about 5% (baseline RMSE ≈ 0.365 in log wage). (2) Random forest hyperparameters were tuned once on model B and applied to all models, which may favour models including exposure; the linear model is therefore the reference for comparisons D–E and F–G. (3) The placebo was drawn once (correlation with true exposure: −0.08).